In [41]:
import fastf1
import plotly.graph_objects as go
import yaml
import pandas as pd

with open("C:/Users/boncl/Repos/simply-lovely/config/teams_drivers_info.yaml", "r") as file:
    config = yaml.safe_load(file)
fastf1.Cache.enable_cache('C:/Users/boncl/Repos/simply-lovely/fastf1_cache')

compound_colours = {
    "SOFT": "#E10600",
    "MEDIUM": "#FFD12E",
    "HARD": "#FAF9F6",
    "INTERMEDIATE": "#43B02A",
    "WET": "#0067AD"
}

In [42]:
silverstone_quali_laps = fastf1.get_session(2026, 'Silverstone', 'Q')
silverstone_quali_laps.load()
quali_laps = pd.DataFrame(silverstone_quali_laps.laps.pick_quicklaps())
quali_laps

core           INFO 	Loading data for British Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['12', '16', '44', '63', '6', '1', '3', '81', '41', '30', '5', '10', '27', '87', '55', '23', '31', '77', '43', '11', '18', '14']


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
1,0 days 00:21:03.056000,ANT,12,0 days 00:01:29.719000,2.0,1.0,NaT,NaT,0 days 00:00:28.836000,0 days 00:00:36.103000,...,False,Mercedes,0 days 00:19:33.337000,2026-07-04 15:06:29.149,12,NaN,False,,False,True
4,0 days 00:28:54.146000,ANT,12,0 days 00:01:30.261000,5.0,2.0,NaT,NaT,0 days 00:00:28.857000,0 days 00:00:36.484000,...,False,Mercedes,0 days 00:27:23.885000,2026-07-04 15:14:19.697,1,NaN,False,,False,True
8,0 days 00:53:52.536000,ANT,12,0 days 00:01:29.043000,9.0,3.0,NaT,NaT,0 days 00:00:28.692000,0 days 00:00:35.864000,...,True,Mercedes,0 days 00:52:23.493000,2026-07-04 15:39:19.305,1,NaN,False,,False,True
11,0 days 01:01:17.660000,ANT,12,0 days 00:01:28.493000,12.0,4.0,NaT,NaT,0 days 00:00:28.292000,0 days 00:00:35.731000,...,True,Mercedes,0 days 00:59:49.167000,2026-07-04 15:46:44.979,1,NaN,False,,False,True
14,0 days 01:13:39.562000,ANT,12,0 days 00:01:28.385000,15.0,5.0,NaT,NaT,0 days 00:00:28.398000,0 days 00:00:35.547000,...,True,Mercedes,0 days 01:12:11.177000,2026-07-04 15:59:06.989,1,NaN,False,,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,0 days 00:17:03.856000,STR,18,0 days 00:01:33.835000,2.0,1.0,NaT,NaT,0 days 00:00:29.667000,0 days 00:00:38.045000,...,True,Aston Martin,0 days 00:15:30.021000,2026-07-04 15:02:25.833,1,NaN,False,,False,True
295,0 days 00:22:50.523000,STR,18,0 days 00:01:33.577000,5.0,2.0,NaT,NaT,0 days 00:00:29.683000,0 days 00:00:37.869000,...,True,Aston Martin,0 days 00:21:16.946000,2026-07-04 15:08:12.758,1,NaN,False,,False,True
298,0 days 00:30:56.204000,STR,18,0 days 00:01:32.863000,8.0,3.0,NaT,NaT,0 days 00:00:29.380000,0 days 00:00:37.793000,...,True,Aston Martin,0 days 00:29:23.341000,2026-07-04 15:16:19.153,12,NaN,False,,False,True
304,0 days 00:23:48.758000,ALO,14,0 days 00:01:33.025000,5.0,2.0,NaT,NaT,0 days 00:00:29.718000,0 days 00:00:37.675000,...,True,Aston Martin,0 days 00:22:15.733000,2026-07-04 15:09:11.545,1,NaN,False,,False,True


In [43]:
silverstone_quali_status = pd.DataFrame(silverstone_quali_laps.session_status)
silverstone_quali_session = quali_laps.merge(silverstone_quali_status, on=['Time'], how='outer')
silverstone_quali_session['Status'] = silverstone_quali_session['Status'].ffill()

In [44]:
silverstone_quali_session = silverstone_quali_session.copy()

# LapTime is required to draw a rectangle
silverstone_quali_session['LapTime'] = pd.to_timedelta(
    silverstone_quali_session['LapTime'],
    errors='coerce'
)

# LapStartTime is also required
silverstone_quali_session['LapStartTime'] = pd.to_timedelta(
    silverstone_quali_session['LapStartTime'],
    errors='coerce'
)

# Keep only rows that actually represent a timed lap
plot_laps = silverstone_quali_session.dropna(
    subset=['LapTime', 'LapStartTime']
).copy()

plot_laps['LapEndTime'] = (
    plot_laps['LapStartTime'] +
    plot_laps['LapTime']
)

status_changes = (
    silverstone_quali_session[['Time', 'Status']]
    .drop_duplicates()
    .sort_values('Time')
)

status_changes = status_changes[
    status_changes['Status'] != status_changes['Status'].shift()
]

gaps = []

finish_time = None

for _, row in status_changes.iterrows():

    if row['Status'] == 'Finished':
        finish_time = pd.Timedelta(row['Time'])

    elif row['Status'] == 'Started' and finish_time is not None:

        # Laps that started before the official session finished
        session_laps = plot_laps[
            plot_laps['LapStartTime'] <= finish_time
        ]

        # The latest time any of those laps actually finished
        actual_finish_time = session_laps['LapEndTime'].max()

        gaps.append(
            (actual_finish_time, pd.Timedelta(row['Time']))
        )

        finish_time = None

def compress_time(time):

    time = pd.Timedelta(time)

    offset = pd.Timedelta(0)

    for gap_start, gap_end in gaps:

        if time <= gap_start:
            break

        offset += min(time, gap_end) - gap_start

    return time - offset

plot_laps['CompressedLapStartTime'] = (
    plot_laps['LapStartTime']
    .apply(compress_time)
)

# Convert to numeric seconds for Plotly
plot_laps['CompressedLapStartSeconds'] = (
    plot_laps['CompressedLapStartTime']
    .dt.total_seconds()
)

In [45]:
fig4 = go.Figure()

# Fastest valid qualifying lap
fastest_quali_lap = plot_laps['LapTime'].min()

# Drivers with at least one valid timed lap
drivers = plot_laps['Driver'].unique()

# Numerical y position for each driver
driver_positions = {
    driver: i for i, driver in enumerate(drivers)
}

for driver in drivers:

    driver_quali_laps = plot_laps[
        plot_laps['Driver'] == driver
    ]

    driver_fastest_quali_lap = driver_quali_laps['LapTime'].min()

    y_pos = driver_positions[driver]

    for _, lap in driver_quali_laps.iterrows():

        # Decide rectangle colour
        if lap['LapTime'] == fastest_quali_lap:
            colour = 'purple'

        elif lap['LapTime'] == driver_fastest_quali_lap:
            colour = 'green'

        else:
            colour = 'yellow'

        # Start in compressed session time
        start_time = lap['CompressedLapStartSeconds']

        # LapTime is a Timedelta -> convert to seconds
        lap_duration = lap['LapTime'].total_seconds()

        end_time = start_time + lap_duration

        fig4.add_trace(
            go.Scatter(
                x=[
                    start_time,
                    end_time,
                    end_time,
                    start_time,
                    start_time
                ],

                y=[
                    y_pos - 0.35,
                    y_pos - 0.35,
                    y_pos + 0.35,
                    y_pos + 0.35,
                    y_pos - 0.35
                ],

                mode='lines',
                fill='toself',
                fillcolor=colour,
                line=dict(
                    color=colour,
                    width=1
                ),
                showlegend=False,

                hovertemplate=(
                    f"Driver: {driver}<br>"
                    f"Lap: {lap['LapNumber']}<br>"
                    f"Lap time: {lap['LapTime']}<extra></extra>"
                )
            )
        )

qualifying_starts = (
    status_changes[
        status_changes['Status'] == 'Started'
    ]['Time']
    .apply(pd.Timedelta)
    .tolist()
)

tickvals = [
    compress_time(start).total_seconds()
    for start in qualifying_starts[:3]
]

fig4.update_layout(
    height=700,
    xaxis=dict(
        title='Qualifying Session Time',
        tickmode='array',
        tickvals=tickvals,
        ticktext=[
            'Q1 Start',
            'Q2 Start',
            'Q3 Start'
        ],
        showgrid=True
    ),
    yaxis=dict(
        tickmode='array',
        tickvals=list(driver_positions.values()),
        ticktext=list(driver_positions.keys()),
        autorange='reversed'
    ),
    title = dict(text = "Silverstone Quali Plot", font = dict( size = 20)
))

fig4.show()

In [ ]:
#TODO: the filtering of quali laps by the drivers selected by the user

quali_laps_all = silverstone_quali_laps.laps.dropna(subset=["LapTime"])
fastest_by_driver = quali_laps_all.sort_values("LapTime").drop_duplicates(subset="Driver", keep = "first").sort_values("LapTime")

fig5 = go.Figure()
hover_template = "Lap Time: %{customdata} s<br>Velocity: %{x} m/s<br>Distance: %{y} m"
for driver in fastest_by_driver['Driver']:
    driver_lap = fastest_by_driver[fastest_by_driver['Driver'] == driver]
    driver_team = driver_lap['Team'].min()
    driver_colour = config['teams'][driver_team]['drivers'][driver]['colour']
    laptime = driver_lap['LapTime'].dt.total_seconds()
    telemetry = driver_lap.get_telemetry()
    fig5.add_trace(
        go.Scatter(
            x=telemetry['Distance'],
            y=telemetry['Speed'],
            name=driver,
            mode='lines',
            line=dict(color=f"rgba({driver_colour})"),
            hovertemplate = hover_template,
            customdata=[laptime]*len(telemetry['Distance'])))

fig5.update_layout(title="Fastest Quali Laps Comparison", xaxis_title="Distance (m)", yaxis_title="Velocity (m/s)")
fig5.show()